In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)




Libraries imported successfully.


In [2]:


DATASET_PATH = "Loan_Default_Cleaned.csv"

df = pd.read_csv(DATASET_PATH)

print("Dataset Shape:", df.shape)
display(df.head())


Dataset Shape: (255347, 18)


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [3]:


print("Default Value Counts:")
print(df["Default"].value_counts())

print("\nDefault Percentage:")
print((df["Default"].value_counts(normalize=True) * 100).round(2))


Default Value Counts:
Default
0    225694
1     29653
Name: count, dtype: int64

Default Percentage:
Default
0    88.39
1    11.61
Name: proportion, dtype: float64


In [4]:


drop_columns = ["Default"]


if "LoanID" in df.columns:
    drop_columns.append("LoanID")

X = df.drop(columns=drop_columns)
y = df["Default"]

print("Features:", X.shape)
print("Target:", y.shape)


Features: (255347, 16)
Target: (255347,)


In [5]:


numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical Columns:")
print(numerical_cols)

print("\nCategorical Columns:")
print(categorical_cols)


Numerical Columns:
['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']

Categorical Columns:
['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']


In [6]:


numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

transformers = []

if numerical_cols:
    transformers.append(
        ("num", numeric_transformer, numerical_cols)
    )

if categorical_cols:
    transformers.append(
        ("cat", categorical_transformer, categorical_cols)
    )

preprocessor = ColumnTransformer(
    transformers=transformers
)

print("Preprocessing pipeline created.")


Preprocessing pipeline created.


In [7]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Records:", len(X_train))
print("Testing Records :", len(X_test))


Training Records: 204277
Testing Records : 51070


In [8]:


model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Random Forest model created.")


Random Forest model created.


In [9]:


print("Training model...")

pipeline.fit(X_train, y_train)

print("Model training completed.")


Training model...
Model training completed.


In [10]:


train_pred = pipeline.predict(X_train)
test_pred = pipeline.predict(X_test)


test_probability = pipeline.predict_proba(X_test)[:, 1]

print("Predictions generated.")


Predictions generated.


## Model Evaluation




In [11]:


test_accuracy = accuracy_score(y_test, test_pred)

test_precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_pred,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    test_probability
)

metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Score": [
        test_accuracy,
        test_precision,
        test_recall,
        test_f1,
        test_roc_auc
    ]
})

metrics["Score"] = metrics["Score"].round(4)

display(metrics)


,Metric,Score
0,Accuracy,0.7305
1,Precision,0.2446
2,Recall,0.6324
3,F1 Score,0.3528
4,ROC-AUC,0.7539


In [12]:


print(classification_report(
    y_test,
    test_pred,
    zero_division=0
))


              precision    recall  f1-score   support

           0       0.94      0.74      0.83     45139
           1       0.24      0.63      0.35      5931

    accuracy                           0.73     51070
   macro avg       0.59      0.69      0.59     51070
weighted avg       0.86      0.73      0.77     51070



In [13]:


cm = confusion_matrix(y_test, test_pred)

print("Confusion Matrix:")
print(cm)

if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()

    print("\nTrue Negative :", tn)
    print("False Positive:", fp)
    print("False Negative:", fn)
    print("True Positive :", tp)


Confusion Matrix:
[[33557 11582]
 [ 2180  3751]]

True Negative : 33557
False Positive: 11582
False Negative: 2180
True Positive : 3751


## Overfitting / Underfitting Check




In [14]:


train_accuracy = accuracy_score(y_train, train_pred)
train_precision = precision_score(y_train, train_pred, zero_division=0)
train_recall = recall_score(y_train, train_pred, zero_division=0)
train_f1 = f1_score(y_train, train_pred, zero_division=0)

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Training": [
        train_accuracy,
        train_precision,
        train_recall,
        train_f1
    ],
    "Testing": [
        test_accuracy,
        test_precision,
        test_recall,
        test_f1
    ]
})

comparison["Training"] = comparison["Training"].round(4)
comparison["Testing"] = comparison["Testing"].round(4)

display(comparison)


,Metric,Training,Testing
0,Accuracy,0.7494,0.7305
1,Precision,0.2742,0.2446
2,Recall,0.7031,0.6324
3,F1 Score,0.3945,0.3528


In [15]:


accuracy_difference = train_accuracy - test_accuracy

print("Training Accuracy:", round(train_accuracy, 4))
print("Testing Accuracy :", round(test_accuracy, 4))
print("Difference       :", round(accuracy_difference, 4))

if accuracy_difference > 0.10:
    result = "POSSIBLE OVERFITTING"
elif train_accuracy < 0.70 and test_accuracy < 0.70:
    result = "POSSIBLE UNDERFITTING"
else:
    result = "MODEL GENERALIZES REASONABLY WELL"

print("\nResult:", result)


Training Accuracy: 0.7494
Testing Accuracy : 0.7305
Difference       : 0.0189

Result: MODEL GENERALIZES REASONABLY WELL


In [16]:


results = pd.DataFrame([{
    "Training Accuracy": train_accuracy,
    "Testing Accuracy": test_accuracy,
    "Testing Precision": test_precision,
    "Testing Recall": test_recall,
    "Testing F1 Score": test_f1,
    "Testing ROC-AUC": test_roc_auc,
    "Train-Test Accuracy Difference": accuracy_difference,
    "Overfitting/Underfitting Result": result
}])

results.to_csv(
    "loan_model_evaluation_results.csv",
    index=False
)

print("Saved: loan_model_evaluation_results.csv")


Saved: loan_model_evaluation_results.csv
